# Credit card fraud on the ULB benchmark

A quick walk through the dataset, three classifiers, and the cost-sensitive threshold sweep
that reframes the whole evaluation. The pipeline also lives as a single script at
`src/run_analysis.py` — this notebook is the narrative version.

Before you run it: place the Kaggle `creditcard.csv` at `data/creditcard.csv` relative to
the repo root.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RNG = 42
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})

## 1. Load and profile

The first thing to notice is the class balance. At 0.17 percent positive, a trivial
classifier predicting "not fraud" for every row already clears 99.8 percent accuracy.

In [ ]:
df = pd.read_csv("data/creditcard.csv")
print(f"rows: {len(df):,}")
print(f"positives: {int(df['Class'].sum()):,}")
print(f"positive rate: {df['Class'].mean() * 100:.4f}%")
df.head()

In [ ]:
counts = df["Class"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(["Legitimate", "Fraud"], counts.values, color=["#4c72b0", "#c44e52"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_yscale("log")
ax.set_title(f"{counts.iloc[1]:,} fraud out of {counts.sum():,} ({counts.iloc[1] / counts.sum() * 100:.3f}%)")
plt.tight_layout()

## 2. The PCA anonymisation

Features V1 through V28 are principal components. I cannot tell you what V14 or V17
represent in the original feature space; the release agreement prevents that. What I can
tell you is that the PCA components preserve enough structure to train a useful classifier,
which is the thing that matters.

`Time` is seconds-since-first-transaction. `Amount` is the transaction amount in EUR.
Both stay in original units and get standardised before modelling.

In [ ]:
X = df.drop(columns=["Class"]).copy()
y = df["Class"].values
scaler = StandardScaler()
X[["Time", "Amount"]] = scaler.fit_transform(X[["Time", "Amount"]])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG,
)
print(f"train positives: {y_train.sum()}   test positives: {y_test.sum()}")

## 3. Three models on the same split

Logistic regression with balanced class weights. XGBoost with `scale_pos_weight`.
Isolation Forest trained only on the legitimate class. The Isolation Forest is the
unsupervised baseline — it exists to show how much label-carried information the
supervised models use.

In [ ]:
logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG).fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

scale_pos = (y_train == 0).sum() / max(1, (y_train == 1).sum())
xgb = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.1, scale_pos_weight=scale_pos,
    eval_metric="aucpr", tree_method="hist", random_state=RNG, n_jobs=-1,
).fit(X_train, y_train)
xgb_scores = xgb.predict_proba(X_test)[:, 1]

iforest = IsolationForest(
    n_estimators=200, contamination=float(y_train.mean()), random_state=RNG, n_jobs=-1,
).fit(X_train[y_train == 0])
iforest_raw = -iforest.score_samples(X_test)
iforest_scores = (iforest_raw - iforest_raw.min()) / (iforest_raw.max() - iforest_raw.min())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, scores, color in [
    ("Logistic regression", logreg_scores, "#4c72b0"),
    ("XGBoost", xgb_scores, "#c44e52"),
    ("Isolation Forest", iforest_scores, "#55a868"),
]:
    p, r, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    ax.plot(r, p, lw=2, color=color, label=f"{name} (AP={ap:.3f})")
ax.axhline(y_test.mean(), color="gray", ls=":", label=f"Baseline={y_test.mean():.4f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend()
ax.set_title("Precision-recall, test set")
plt.tight_layout()

## 4. The threshold is a policy decision

Here's the step most tutorials skip. I set a cost model (missed fraud = 100 EUR,
false positive = 5 EUR) and sweep the threshold. The minimum-cost operating point for
each model is the place a real fraud team would actually deploy it.

You can change `FN_COST` and `FP_COST` below and re-run to see how the ranking shifts.

In [ ]:
FN_COST, FP_COST = 100.0, 5.0
thresholds = np.linspace(0.001, 0.999, 200)

def cost_sweep(scores):
    rows = []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
        rows.append({
            "threshold": t, "tp": tp, "fp": fp, "fn": fn,
            "recall": tp / (tp + fn) if (tp + fn) else 0,
            "total_cost": FN_COST * fn + FP_COST * fp,
        })
    return pd.DataFrame(rows)

sweeps = {
    "Logistic": cost_sweep(logreg_scores),
    "XGBoost":  cost_sweep(xgb_scores),
    "Isolation": cost_sweep(iforest_scores),
}
summary = []
for name, s in sweeps.items():
    best = s.loc[s["total_cost"].idxmin()]
    summary.append({
        "model": name, "threshold": round(best["threshold"], 3),
        "recall": round(best["recall"], 3),
        "missed_fraud": int(best["fn"]), "false_review": int(best["fp"]),
        "total_cost_eur": round(best["total_cost"], 0),
    })
pd.DataFrame(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, s, color in zip(sweeps.keys(), sweeps.values(), ["#4c72b0", "#c44e52", "#55a868"]):
    ax.plot(s["threshold"], s["total_cost"], label=name, color=color, lw=2)
ax.set_xlabel("Threshold"); ax.set_ylabel("Expected cost (EUR)")
ax.set_title("Total cost vs. threshold for each model")
ax.legend()
plt.tight_layout()

## 5. What this does not prove

The PCA anonymisation blocks any feature-importance analysis that means something.
The cost model is illustrative — any institution should plug in its own numbers.
And the test set has 123 positives, so the recall estimates carry a few percentage points
of noise. A cross-validated version of the cost sweep is the honest next step.

For the full write-up, see the [blog post](https://ndjstn.github.io/posts/credit-card-fraud-default-threshold/)
and `REPORT.md` in this repo.